# System Monitoring DTOs

> Standardized telemetry DTOs for the system-monitor capability — the data nouns a monitor tool capability emits (host CPU/RAM + aggregated GPU stats, and per-process GPU usage) and the substrate scheduler consumes for resource-derived admission + GPU subtree attribution.

**Canonical home (execution stage 8 — Option C / PILLAR 1c).** `SystemStats` / `ProcessStats` live here in `cjm-capability-primitives` so the system-monitor tool capability depends only on these data nouns, never on adapter machinery. The monitor is the one migrated capability with **no task adapter** — live telemetry has no cacheable, content-addressable result — so it is consumed through the substrate's *native-dispatch* channel (`PluginManager._get_global_stats` → admission; `JobQueue` GPU subtree attribution via `list_processes`), not the task channel. `MonitorToolProtocol` documents that native surface so future platform monitors (Intel / AMD / Apple Silicon) implement one contract.

**Wire registration is forward-consistency, not an active path.** Today the monitor crosses the worker boundary via bespoke endpoints (`/get_system_status`, `/list_processes`) that return raw `to_dict()`. Registering these DTOs as `@wire_type` keeps every `cjm-capability-primitives` noun uniformly wire-serializable and readies a future unification of those endpoints onto the typed wire layer (the CR-3-era monitor predated it); nothing wire-encodes them yet.

This replaces the dissolving `cjm-infra-plugin-system` (`MonitorPlugin` ABC + these DTOs); the `details` field and the `execute(command=...)` dispatcher are dropped (the CR-3 end-state — their only consumer was the descoped job-monitor GUI).

In [ ]:
#| default_exp monitoring

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Protocol, runtime_checkable

from cjm_plugin_system.core.wire import wire_type

In [ ]:
#| export
@wire_type("monitoring.system_stats")
@dataclass
class SystemStats:
    """Standardized snapshot of host + GPU resources (the scheduler's admission input)."""
    # CPU
    cpu_percent: float = 0.0           # Overall CPU utilization percentage
    # Memory (RAM admission checks)
    memory_used_mb: float = 0.0        # Currently used system RAM in MB
    memory_total_mb: float = 0.0       # Total system RAM in MB
    memory_available_mb: float = 0.0   # Available system RAM in MB
    # GPU (VRAM admission checks; aggregated across all visible devices)
    gpu_type: str = "None"             # GPU vendor: 'NVIDIA', 'AMD', 'Intel', 'None'
    gpu_free_memory_mb: float = 0.0    # Free GPU memory in MB
    gpu_total_memory_mb: float = 0.0   # Total GPU memory in MB
    gpu_used_memory_mb: float = 0.0    # Used GPU memory in MB
    gpu_load_percent: float = 0.0      # GPU compute utilization percentage

    def to_dict(self) -> Dict[str, Any]:  # Serialized representation
        """Convert to dictionary for JSON serialization."""
        return asdict(self)

In [ ]:
#| export
@wire_type("monitoring.process_stats")
@dataclass
class ProcessStats:
    """Per-process GPU usage, reported by a monitor's `list_processes` (GPU subtree attribution)."""
    pid: int = 0                # OS process ID
    gpu_index: int = -1         # GPU index (0-based); -1 if not GPU-bound or unknown
    gpu_memory_mb: float = 0.0  # GPU memory attributable to this process, in MB
    command: str = ""           # Process command line (or short name)

    def to_dict(self) -> Dict[str, Any]:  # Serialized representation
        """Convert to dictionary for JSON serialization."""
        return asdict(self)

In [ ]:
#| export
@runtime_checkable
class MonitorToolProtocol(Protocol):
    """The native surface a system-monitor tool capability exposes.

    The substrate consumes a monitor through this surface by NAME (duck-typed,
    host-no-imports per CR-1) — `get_system_status` feeds resource-derived
    admission; `list_processes` feeds per-worker GPU subtree attribution. There
    is no task adapter: this is the native-dispatch contract, and the manifest's
    `structural_surface` is what a future auto-detect could match against. Platform
    monitors (NVIDIA today; Intel / AMD / Apple Silicon later) each implement it.
    """
    def get_system_status(self) -> SystemStats: ...      # Current host + aggregated GPU telemetry
    def list_processes(self) -> List[ProcessStats]: ...  # Per-process GPU usage ([] if no per-process visibility)

In [ ]:
# Test SystemStats
s = SystemStats(cpu_percent=12.5, memory_used_mb=8000.0, memory_total_mb=32000.0,
                memory_available_mb=24000.0, gpu_type="NVIDIA", gpu_free_memory_mb=6000.0,
                gpu_total_memory_mb=24000.0, gpu_used_memory_mb=18000.0, gpu_load_percent=42.0)
print(s.to_dict())
assert s.gpu_type == "NVIDIA" and s.gpu_free_memory_mb == 6000.0
# Default (no-GPU) snapshot
assert SystemStats().gpu_type == "None"

In [ ]:
# Test ProcessStats (and the empty default for monitors without per-process visibility)
p = ProcessStats(pid=4242, gpu_index=0, gpu_memory_mb=1536.0, command="python -m worker")
print(p.to_dict())
assert p.pid == 4242 and p.gpu_index == 0
assert ProcessStats().gpu_index == -1

In [ ]:
# Wire-format executable test: both DTOs round-trip TYPED through the simulated
# worker boundary (encode -> json -> decode). Registration is forward-consistency
# (the live monitor transport returns raw to_dict() via bespoke endpoints), but
# keeping them wire-stable is verified here.
import json as _json
from cjm_plugin_system.core.wire import wire_encode, wire_decode, WIRE_KIND_KEY

s = SystemStats(cpu_percent=5.0, gpu_type="NVIDIA", gpu_free_memory_mb=1000.0)
env_s = wire_encode(s)
assert env_s[WIRE_KIND_KEY] == "monitoring.system_stats"
back_s = wire_decode(_json.loads(_json.dumps(env_s)))
assert isinstance(back_s, SystemStats) and back_s == s

p = ProcessStats(pid=99, gpu_index=0, gpu_memory_mb=512.0, command="x")
env_p = wire_encode(p)
assert env_p[WIRE_KIND_KEY] == "monitoring.process_stats"
back_p = wire_decode(_json.loads(_json.dumps(env_p)))
assert isinstance(back_p, ProcessStats) and back_p == p

# MonitorToolProtocol is runtime_checkable: a class with both methods satisfies it.
class _FakeMonitor:
    def get_system_status(self): return SystemStats()
    def list_processes(self): return []
assert isinstance(_FakeMonitor(), MonitorToolProtocol)
print("monitoring DTO wire round-trips + protocol check OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()